# Bài 5 · Series & DataFrame chuyên sâu

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn:

1. Hiểu index là danh tính dòng: `loc`/`iloc` không lẫn nữa, biết alignment tự khớp nhãn.
2. Biến đổi cột theo đúng thứ tự ưu tiên: vector hoá → `map` → `apply`.
3. Dùng `groupby` thành thạo: `agg` đặt tên, `transform`, `pivot_table`.
4. Ghép bảng bằng `concat`/`merge` và **kiểm tra kết quả ghép** như một thói quen.

Dữ liệu: Santiago (2 snapshot: 09/2025 và 06/2026).

In [ ]:
import pandas as pd

BASE = "https://data.insideairbnb.com/chile/rm/santiago"
df = pd.read_csv(f"{BASE}/2026-06-29/visualisations/listings.csv")   # snapshot mới
t9 = pd.read_csv(f"{BASE}/2025-09-27/visualisations/listings.csv")   # snapshot cũ
print("06/2026:", df.shape, "| 09/2025:", t9.shape)

## 1. Index: danh tính của dòng

In [ ]:
# set_index: tra cứu phòng theo id thay vì "dòng thứ mấy"
d = df.set_index("id")
mot_id = d.index[0]
d.loc[mot_id, ["neighbourhood", "room_type", "price"]]

In [ ]:
# loc theo NHÃN vs iloc theo VỊ TRÍ — ví dụ dồn về một slide
s = pd.Series([10, 20, 30], index=[2, 0, 1])
print("loc[2]  =", s.loc[2], "   (dòng mang nhãn 2)")
print("iloc[2] =", s.iloc[2], "   (dòng thứ ba)")

In [ ]:
# Alignment: phép toán giữa 2 Series tự khớp theo index
gia_t9  = t9.groupby("neighbourhood")["price"].median()
gia_t6  = df.groupby("neighbourhood")["price"].median()

thay_doi_pct = ((gia_t6 - gia_t9) / gia_t9 * 100).round(1)
thay_doi_pct.sort_values(ascending=False).head()

In [ ]:
# Bẫy của alignment: quận chỉ có ở MỘT snapshot -> NaN lặng lẽ
print("Số quận NaN sau phép chia:", thay_doi_pct.isna().sum())
print("Đó là:", list(thay_doi_pct[thay_doi_pct.isna()].index))

Quy tắc từ buổi này: **sau mọi phép toán giữa hai bảng, đếm NaN mới sinh ra** —
mỗi NaN là một nhãn không khớp cần được giải thích.

## 2. map & apply — ba nấc biến đổi

Ưu tiên: (1) vector hoá có sẵn → (2) `map` → (3) `apply(axis=1)`. Xuống nấc dưới chỉ khi nấc trên
không diễn đạt nổi.

In [ ]:
# map với dict: Việt hoá nhãn loại phòng
viet_hoa = {"Entire home/apt": "Nguyên căn", "Private room": "Phòng riêng",
            "Shared room": "Phòng chung", "Hotel room": "Khách sạn"}
df["loai"] = df["room_type"].map(viet_hoa)
df["loai"].value_counts()

In [ ]:
# map với hàm: clean_price của buổi 2 quay lại (bảng listings ĐẦY ĐỦ có giá dạng chuỗi thế này)
def clean_price(s):
    try:
        return float(str(s).replace("$", "").replace(",", ""))
    except ValueError:
        return None

gia_chuoi = pd.Series(["$45,647.00", "N/A", "$19,856.00", None])
gia_chuoi.map(clean_price)

In [ ]:
import time

# apply(axis=1) vs vector hoá — đo hẳn hoi
t0 = time.perf_counter()
kq1 = df.apply(lambda r: r["number_of_reviews_ltm"] / r["price"] * 1e4
               if r["price"] and pd.notna(r["price"]) else None, axis=1)
t_apply = time.perf_counter() - t0

t0 = time.perf_counter()
kq2 = df["number_of_reviews_ltm"] / df["price"] * 1e4
t_vec = time.perf_counter() - t0

print(f"apply : {t_apply*1000:7.1f} ms")
print(f"vector: {t_vec*1000:7.1f} ms  (nhanh hơn ~{t_apply/t_vec:.0f} lần)")

## 3. groupby chuyên sâu

In [ ]:
# agg đặt tên + kèm cỡ nhóm — thuốc giải "quán quân 5 phòng"
tk = df.groupby("neighbourhood")["price"].agg(trung_vi="median", so_phong="size")
tk[tk["so_phong"] >= 500].nlargest(5, "trung_vi")

In [ ]:
# groupby nhiều khoá -> MultiIndex; reset_index() khi cần bảng phẳng
df.groupby(["neighbourhood", "loai"])["price"].median().reset_index().head(6)

In [ ]:
# transform: so TỪNG PHÒNG với trung vị quận của nó
df["he_so_gia"] = df["price"] / df.groupby("neighbourhood")["price"].transform("median")

nghi_van = df[df["he_so_gia"] > 20]
print(f"{len(nghi_van)} phòng giá gấp >20 lần trung vị quận mình:")
nghi_van[["name", "neighbourhood", "price", "he_so_gia"]].nlargest(3, "he_so_gia").round(1)

In [ ]:
# pivot_table: bảng chéo quận × loại phòng
df[df["neighbourhood"].isin(["Santiago", "Providencia", "Las Condes", "Ñuñoa"])].pivot_table(
    values="price", index="neighbourhood", columns="loai", aggfunc="median")

## 4. concat & merge

In [ ]:
# concat: chồng 2 snapshot — LUÔN khai cột nguồn trước khi chồng
t9["snapshot"], df["snapshot"] = "2025-09", "2026-06"
ca_hai = pd.concat([t9, df], ignore_index=True)

ca_hai.groupby("snapshot").agg(so_phong=("id", "size"),
                               gia_trung_vi=("price", "median"))

In [ ]:
# merge: nâng cấp bảng chính bằng bảng tra cứu nhỏ
vung = pd.DataFrame({
    "neighbourhood": ["Santiago", "Providencia", "Las Condes", "Ñuñoa", "Recoleta"],
    "vung": ["Trung tâm", "Đông", "Đông", "Đông", "Bắc"],
})
m = df.merge(vung, on="neighbourhood", how="left", validate="m:1")

# Thói quen sau merge: số dòng + độ khớp
print("Trước:", len(df), "| Sau:", len(m), "| Chưa khớp vùng:", m["vung"].isna().sum())
m.groupby("vung")["price"].median()

Thử đổi `how="left"` thành `how="inner"` rồi chạy lại cell trên — bao nhiêu dòng "bốc hơi"?
Đó chính là khác biệt giữa hai kiểu join (xem hình trong slide).

## 5. Bài tập tại lớp

### Bài 1 — Quận nóng lạnh

Dùng `thay_doi_pct` (mục 1): tìm 3 quận **tăng giá mạnh nhất** và 3 quận **giảm mạnh nhất**
giữa 2 snapshot, nhưng chỉ xét quận có **≥ 200 phòng ở cả hai** snapshot.

In [ ]:
# TODO Bài 1 (scaffold):
dem_t9 = t9.groupby("neighbourhood").size()
dem_t6 = df.groupby("neighbourhood").size()
du_lon = (dem_t9 >= 200) & (dem_t6 >= 200)     # alignment lại ra tay!

hop_le = thay_doi_pct[du_lon]
print("Tăng mạnh nhất:"); print(hop_le.nlargest(3))
print("\nGiảm mạnh nhất:"); print(hop_le.nsmallest(3))

### Bài 2 — z-score trong nhóm bằng transform

Tạo cột `z_gia`: giá của phòng trừ **mean của quận nó**, chia **std của quận nó**
(z-score nội quận). Đếm số phòng có `|z_gia| > 3`. Vì sao dùng z-score *nội quận* hợp lý hơn
z-score toàn thành phố?

In [ ]:
# TODO Bài 2:
g = df.groupby("neighbourhood")["price"]
df["z_gia"] = (df["price"] - g.transform("mean")) / g.transform("std")
print("Số phòng |z| > 3:", (df["z_gia"].abs() > 3).sum())

### Bài 3 — Một bảng cho sếp

Tạo **một** bảng: mỗi dòng một quận (≥300 phòng), các cột: số phòng, % nguyên căn,
giá trung vị, giá trung vị đổi sang triệu VND (× 28 / 1e6), sắp xếp giảm dần theo giá.
Gợi ý: tạo cột bool `nguyen_can` trước, rồi một chuỗi `groupby().agg()` + lọc + sort.

In [ ]:
# TODO Bài 3:
df["nguyen_can"] = df["room_type"] == "Entire home/apt"
bang = (df.groupby("neighbourhood")
          .agg(so_phong=("id", "size"),
               pct_nguyen_can=("nguyen_can", "mean"),
               gia_trung_vi=("price", "median")))
bang = bang[bang["so_phong"] >= 300].sort_values("gia_trung_vi", ascending=False)
bang["gia_trieu_vnd"] = (bang["gia_trung_vi"] * 28 / 1e6).round(2)
bang.round(3).head(8)

## 6. Bài tập về nhà — "Ai rời thị trường?"

Dùng cột `id` của hai snapshot:

1. Bao nhiêu phòng có ở 09/2025 nhưng **biến mất** ở 06/2026? Bao nhiêu phòng **mới xuất hiện**?
   (gợi ý: set + phép trừ, hoặc `merge(how="outer", indicator=True)` — thử cả hai, đối chiếu!)
2. Nhóm phòng "rời thị trường" có gì khác nhóm "ở lại"? So sánh giá trung vị, loại phòng,
   quận tập trung.
3. Viết 5 dòng nhận xét kiểu báo cáo — mỗi kết luận kèm số liệu.

Đây chính là một phân tích "so sánh giữa snapshot" mà bài tập lớn yêu cầu.

In [ ]:
RUN_CHALLENGE = False

if RUN_CHALLENGE:
    chung = pd.merge(t9[["id", "price", "room_type"]], df[["id"]],
                     on="id", how="outer", indicator=True)
    print(chung["_merge"].value_counts())
    ...

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| `loc` nhãn / `iloc` vị trí; phép toán tự **align** theo index | Đọc đúng và không trừ nhầm dòng |
| NaN mới sau phép toán 2 bảng = nhãn không khớp | Thói quen đếm NaN |
| vector hoá → `map` → `apply` (đúng thứ tự) | Code nhanh và rõ nghĩa |
| `agg` đặt tên + cỡ nhóm; `transform` so dòng với nhóm | Xương sống của KPI + QA ngoại lai |
| Sau merge: kiểm số dòng, đếm NaN, `validate=` | Lỗi merge không có thông báo — chỉ có kết quả sai |

**Buổi sau:** dữ liệu không chỉ nằm trong CSV — API, SQL, Parquet, và một người bạn mới: DuckDB.